# SOEA-Plus V2 — Complete Evaluation Pipeline
## Second-Order Error Awareness Plus: Addressing All Reviewer Concerns

**Version:** 2.0 — Full Revision  
**Dataset:** PubMedQA pqa_labeled — 1000 samples, seed=42  
**Models:** GPT-4.1-mini · Llama-3.3-70b · Gemini-2.5-Flash  

### What this notebook does (addresses ALL reviewer concerns):

| # | Reviewer Concern | Solution in this Notebook |
|---|---|---|
| 1 | Different dataset splits for each model | ALL models use the SAME 1000 samples |
| 2 | Sample size too small (300) | Expanded to 1000 (full PubMedQA human-labeled) |
| 3 | Threshold 0.30 unjustified | Threshold sweep 0.10-0.50 |
| 4 | PDEMC weights unjustified | Sensitivity analysis (6 weight configs) |
| 5 | Single-prompt != true metacognition | Multi-turn protocol comparison |
| 6 | No additional baselines | Always Commit, Always Abstain, Random, Threshold |
| 7 | Dataset construction not described | Full metadata: seed, filtering, label distribution |
| 8 | Control actions not defined in prompt | Explicit definitions added to prompt |
| 9 | No statistical analysis | Bootstrap 95% CI, std dev, per-class accuracy |
| 10 | Only 2 models | 3 models: GPT + Llama + Gemini |
| 11 | Acronyms NLI, ECE not expanded | Fixed in paper (separate step) |

---
## API keys are already configured — just run all cells!

In [ ]:
# CELL 1 — Install all required packages
!pip install -q openai requests datasets pandas numpy matplotlib seaborn tqdm scipy

In [ ]:
# CELL 2 — API KEYS (already filled in)

OPENAI_API_KEY = "YOUR_OPENAI_API_KEY_ybioHhuVT3BlbkFJUYaBe4SFb_OEsSisSt2QiD58L7NEepaDogB42qu-S-GvwnEq1xgVf7ipdydzHjP_pJvVO6U8gA"
GROQ_API_KEY   = "YOUR_GROQ_API_KEY"
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API keys configured successfully")

In [ ]:
# CELL 3 — Imports and global configuration
import json, re, time, os, csv, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from scipy import stats
import requests
from openai import OpenAI

RANDOM_SEED   = 42
N_SAMPLES     = 1000
THRESHOLD     = 0.30
W_ACC, W_MF, W_CR = 0.30, 0.30, 0.40
N_BOOTSTRAP   = 1000

np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
MODEL_COLORS = {
    "GPT-4.1-mini":     "#2196F3",
    "Llama-3.3-70b":    "#FF9800",
    "Gemini-2.5-Flash": "#4CAF50",
}

print("Imports and configuration ready")
print(f"Random seed: {RANDOM_SEED}, N samples: {N_SAMPLES}")
print(f"PDEMC weights: Acc={W_ACC}, MF={W_MF}, CR={W_CR}")

In [ ]:
# CELL 4 — Download PubMedQA (1000 human-labeled samples)
from datasets import load_dataset

print("Downloading PubMedQA pqa_labeled split from HuggingFace...")
dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train", trust_remote_code=True)
print(f"Total available samples: {len(dataset)}")

rows = []
for item in dataset:
    contexts = item.get("context", {})
    if isinstance(contexts, dict):
        passages = contexts.get("contexts", [])
    elif isinstance(contexts, list):
        passages = contexts
    else:
        passages = []
    evidence = " ".join(passages)[:1200] if passages else ""

    label_raw = str(item.get("final_decision", "")).lower().strip()
    if label_raw == "yes":
        label = "SUPPORTED"
    elif label_raw == "no":
        label = "REFUTED"
    else:
        label = "INCONCLUSIVE"

    rows.append({
        "pubmed_id": str(item.get("pubid", len(rows))),
        "claim":     str(item.get("question", "")),
        "evidence":  evidence,
        "gold_label": label
    })

df_full = pd.DataFrame(rows)
df = df_full.sample(n=min(N_SAMPLES, len(df_full)), random_state=RANDOM_SEED).reset_index(drop=True)

print(f"\nDataset ready: {len(df)} samples")
print(f"Source: PubMedQA pqa_labeled | Seed: {RANDOM_SEED} | Filtering: none")
print("\nLabel distribution:")
vc = df["gold_label"].value_counts()
vcp = df["gold_label"].value_counts(normalize=True)
for lbl in ["SUPPORTED", "REFUTED", "INCONCLUSIVE"]:
    print(f"  {lbl:15s}: {vc.get(lbl,0):4d} ({vcp.get(lbl,0)*100:.1f}%)")

metadata = {
    "source": "PubMedQA pqa_labeled split (HuggingFace)",
    "n_total_available": len(df_full),
    "n_samples_used": len(df),
    "random_seed": RANDOM_SEED,
    "selection_method": "random sampling with fixed seed",
    "filtering_criteria": "none - all human-labeled samples included",
    "label_distribution": df["gold_label"].value_counts().to_dict(),
    "label_proportions": df["gold_label"].value_counts(normalize=True).round(4).to_dict(),
    "note": "All models evaluated on identical sample set to ensure fair comparison"
}
with open("dataset_metadata_v2.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Metadata saved to dataset_metadata_v2.json")

In [ ]:
# CELL 5 — Prompt templates (Protocol A: single-prompt, Protocol B: multi-turn)

SINGLE_PROMPT = """You are a biomedical claim verification assistant. Given a CLAIM and PUBMED EVIDENCE, complete THREE sequential tasks.

CLAIM: {claim}

PUBMED EVIDENCE: {evidence}

Complete ALL THREE tasks:

TASK 1 - DECISION: Classify the claim based on the evidence.
  Options: SUPPORTED | REFUTED | INCONCLUSIVE
  - SUPPORTED: The evidence directly supports the claim
  - REFUTED: The evidence directly contradicts the claim
  - INCONCLUSIVE: The evidence is insufficient, ambiguous, or does not address the claim

TASK 2 - MONITORING: Estimate the probability that your Task 1 decision is WRONG.
  Provide a decimal between 0.0 (certain you are correct) and 1.0 (certain you are wrong).
  Consider: quality of evidence, ambiguity, your confidence in reasoning.

TASK 3 - CONTROL: Based on your error probability from Task 2, choose an action.
  Action definitions (use these criteria strictly):
  - COMMIT: Stand by your decision. Use when error_probability < 0.30 (you are confident)
  - ABSTAIN: Withhold judgment due to uncertainty. Use when 0.30 <= error_probability < 0.60
  - SEEK_EVIDENCE: Request additional information. Use when error_probability >= 0.60 and evidence is insufficient
  - REVISE: Change your Task 1 decision. Use only if you realize a clear error in your reasoning

Respond in EXACTLY this format (no extra text, no explanation):
DECISION: [SUPPORTED/REFUTED/INCONCLUSIVE]
ERROR_PROBABILITY: [0.0-1.0]
ACTION: [COMMIT/ABSTAIN/SEEK_EVIDENCE/REVISE]"""

MT_STAGE1_PROMPT = """You are a biomedical claim verification assistant.

CLAIM: {claim}

PUBMED EVIDENCE: {evidence}

TASK: Classify the claim based ONLY on the evidence provided.
  - SUPPORTED: The evidence directly supports the claim
  - REFUTED: The evidence directly contradicts the claim
  - INCONCLUSIVE: The evidence is insufficient, ambiguous, or does not address the claim

Respond in EXACTLY this format:
DECISION: [SUPPORTED/REFUTED/INCONCLUSIVE]"""

MT_STAGE2_PROMPT = """You previously classified a biomedical claim as: {decision}

CLAIM: {claim}
PUBMED EVIDENCE: {evidence}

Now critically review your decision:
- Is your reasoning well-supported by the evidence?
- Did you miss any important information?
- How likely is it that your decision is WRONG?

Respond in EXACTLY this format:
ERROR_PROBABILITY: [0.0-1.0]"""

MT_STAGE3_PROMPT = """You classified a biomedical claim as: {decision}
Your estimated error probability: {error_prob}

CLAIM: {claim}
PUBMED EVIDENCE: {evidence}

Based on your error probability, choose the appropriate control action:
  - COMMIT: Stand by your decision (use when error_probability < 0.30)
  - ABSTAIN: Withhold judgment (use when 0.30 <= error_probability < 0.60)
  - SEEK_EVIDENCE: Request more information (use when error_probability >= 0.60)
  - REVISE: Change your decision (use only if you realize a clear error)

Respond in EXACTLY this format:
ACTION: [COMMIT/ABSTAIN/SEEK_EVIDENCE/REVISE]"""

print("Prompt templates ready")
print(f"Protocol A (single-prompt): {len(SINGLE_PROMPT)} chars")
print("Protocol B (multi-turn): 3 separate prompts")

In [ ]:
# CELL 6 — Helper functions: parsing + metrics

def parse_single_response(text):
    decision   = "INCONCLUSIVE"
    error_prob = 0.5
    action     = "COMMIT"
    if not text:
        return decision, error_prob, action
    t = text.upper()
    for lbl in ["SUPPORTED", "REFUTED", "INCONCLUSIVE"]:
        if f"DECISION: {lbl}" in t or f"DECISION:{lbl}" in t:
            decision = lbl; break
    m = re.search(r"ERROR_PROBABILITY:\s*([0-9]*\.?[0-9]+)", text, re.IGNORECASE)
    if m:
        try: error_prob = max(0.0, min(1.0, float(m.group(1))))
        except: pass
    for act in ["SEEK_EVIDENCE", "ABSTAIN", "REVISE", "COMMIT"]:
        if f"ACTION: {act}" in t or f"ACTION:{act}" in t:
            action = act; break
    return decision, error_prob, action

def parse_stage1(text):
    if not text: return "INCONCLUSIVE"
    t = text.upper()
    for lbl in ["SUPPORTED", "REFUTED", "INCONCLUSIVE"]:
        if lbl in t: return lbl
    return "INCONCLUSIVE"

def parse_stage2(text):
    if not text: return 0.5
    m = re.search(r"ERROR_PROBABILITY:\s*([0-9]*\.?[0-9]+)", text, re.IGNORECASE)
    if not m:
        m = re.search(r"([0-9]*\.?[0-9]+)", text)
    if m:
        try: return max(0.0, min(1.0, float(m.group(1))))
        except: pass
    return 0.5

def parse_stage3(text):
    if not text: return "COMMIT"
    t = text.upper()
    for act in ["SEEK_EVIDENCE", "ABSTAIN", "REVISE", "COMMIT"]:
        if act in t: return act
    return "COMMIT"

def compute_mf(stage1_correct, error_prob):
    actual = 0.0 if stage1_correct else 1.0
    return 1.0 - abs(error_prob - actual)

def compute_cr(stage1_correct, action):
    if stage1_correct:
        return 1.0 if action == "COMMIT" else 0.0
    else:
        return 0.0 if action == "COMMIT" else 1.0

def compute_pdemc(acc, mf, cr, w_acc=W_ACC, w_mf=W_MF, w_cr=W_CR):
    return w_acc * acc + w_mf * mf + w_cr * cr

def bootstrap_ci(values, n_boot=N_BOOTSTRAP, ci=95):
    arr = np.array(values)
    boot_means = [np.mean(np.random.choice(arr, size=len(arr), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(boot_means, (100 - ci) / 2)
    hi = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return lo, hi

print("Helper functions ready")

In [ ]:
# CELL 7 — API call functions for all 3 models

client_openai = OpenAI(api_key=OPENAI_API_KEY)

def call_gpt(prompt, max_tokens=100, max_retries=4):
    for attempt in range(max_retries):
        try:
            resp = client_openai.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=max_tokens
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            wait = 15 * (attempt + 1)
            if "429" in err or "rate" in err.lower():
                print(f"  GPT rate limit, waiting {wait}s...")
            else:
                print(f"  GPT error: {err[:60]}")
            time.sleep(wait)
    return None

def call_llama(prompt, max_tokens=100, max_retries=4):
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                "https://api.groq.com/openai/v1/chat/completions",
                headers={"Authorization": f"Bearer {GROQ_API_KEY}",
                         "Content-Type": "application/json"},
                json={"model": "llama-3.3-70b-versatile",
                      "messages": [{"role": "user", "content": prompt}],
                      "temperature": 0.0, "max_tokens": max_tokens},
                timeout=30
            )
            if resp.status_code == 200:
                return resp.json()["choices"][0]["message"]["content"].strip()
            elif resp.status_code == 429:
                wait = 20 * (attempt + 1)
                print(f"  Llama rate limit, waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"  Llama error {resp.status_code}: {resp.text[:60]}")
                time.sleep(5)
        except Exception as e:
            print(f"  Llama exception: {str(e)[:60]}")
            time.sleep(5)
    return None

def call_gemini(prompt, max_tokens=100, max_retries=4):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={GEMINI_API_KEY}"
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                url,
                json={"contents": [{"parts": [{"text": prompt}]}],
                      "generationConfig": {"temperature": 0.0, "maxOutputTokens": max_tokens}},
                timeout=30
            )
            if resp.status_code == 200:
                data = resp.json()
                return data["candidates"][0]["content"]["parts"][0]["text"].strip()
            elif resp.status_code == 429:
                wait = 20 * (attempt + 1)
                print(f"  Gemini rate limit, waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"  Gemini error {resp.status_code}: {resp.text[:80]}")
                time.sleep(5)
        except Exception as e:
            print(f"  Gemini exception: {str(e)[:60]}")
            time.sleep(5)
    return None

MODEL_CALL_FN = {
    "GPT-4.1-mini":     call_gpt,
    "Llama-3.3-70b":    call_llama,
    "Gemini-2.5-Flash": call_gemini,
}
MODEL_SLEEP = {
    "GPT-4.1-mini":     0.3,
    "Llama-3.3-70b":    0.5,
    "Gemini-2.5-Flash": 0.5,
}

print("API call functions ready for 3 models:")
for m in MODEL_CALL_FN: print(f"  - {m}")

In [ ]:
# CELL 8 — Protocol A: Single-prompt evaluation (3 models x 1000 samples)
# Saves results incrementally — safe to resume if interrupted

FIELDNAMES_A = ["pubmed_id", "gold_label", "decision", "error_prob", "action",
                "stage1_correct", "mf", "cr", "raw_response"]

def run_single_prompt(model_name, call_fn, sleep_sec):
    safe_name = model_name.replace("-","_").replace(".","_")
    out_file = f"results_A_{safe_name}.csv"

    done_ids = set()
    if os.path.exists(out_file):
        existing = pd.read_csv(out_file)
        done_ids = set(existing["pubmed_id"].astype(str))
        print(f"  Resuming {model_name}: {len(done_ids)} already done")

    write_header = not os.path.exists(out_file)
    with open(out_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES_A)
        if write_header:
            writer.writeheader()

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Protocol A - {model_name}"):
            pid = str(row["pubmed_id"])
            if pid in done_ids:
                continue

            prompt = SINGLE_PROMPT.format(
                claim=row["claim"],
                evidence=row["evidence"][:1000]
            )
            response = call_fn(prompt, max_tokens=80)
            decision, error_prob, action = parse_single_response(response)
            s1_correct = int(decision == row["gold_label"])
            mf = compute_mf(s1_correct, error_prob)
            cr = compute_cr(s1_correct, action)

            writer.writerow({
                "pubmed_id":     pid,
                "gold_label":    row["gold_label"],
                "decision":      decision,
                "error_prob":    round(error_prob, 4),
                "action":        action,
                "stage1_correct": s1_correct,
                "mf":            round(mf, 4),
                "cr":            round(cr, 4),
                "raw_response":  str(response)[:200] if response else ""
            })
            f.flush()
            time.sleep(sleep_sec)

    result_df = pd.read_csv(out_file)
    acc  = result_df['stage1_correct'].mean()
    mf_m = result_df['mf'].mean()
    cr_m = result_df['cr'].mean()
    pdemc = compute_pdemc(acc, mf_m, cr_m)
    print(f"  {model_name} done: {len(result_df)} samples")
    print(f"     Acc={acc:.4f}  MF={mf_m:.4f}  CR={cr_m:.4f}  PDEMC={pdemc:.4f}")
    return out_file

print("Starting Protocol A (Single-Prompt) for all 3 models...")
print("Estimated time: 60-90 minutes total\n")

files_A = {}
for model_name, call_fn in MODEL_CALL_FN.items():
    files_A[model_name] = run_single_prompt(model_name, call_fn, MODEL_SLEEP[model_name])

print("\nProtocol A complete for all 3 models!")

In [ ]:
# CELL 9 — Protocol B: Multi-turn sequential evaluation (3 models x 200 samples)

df_mt = df.head(200).reset_index(drop=True)

FIELDNAMES_B = ["pubmed_id", "gold_label", "decision", "error_prob", "action",
                "stage1_correct", "mf", "cr", "raw_s1", "raw_s2", "raw_s3"]

def run_multi_turn(model_name, call_fn, sleep_sec):
    safe_name = model_name.replace("-","_").replace(".","_")
    out_file = f"results_B_{safe_name}.csv"

    done_ids = set()
    if os.path.exists(out_file):
        existing = pd.read_csv(out_file)
        done_ids = set(existing["pubmed_id"].astype(str))
        print(f"  Resuming {model_name} multi-turn: {len(done_ids)} done")

    write_header = not os.path.exists(out_file)
    with open(out_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES_B)
        if write_header:
            writer.writeheader()

        for _, row in tqdm(df_mt.iterrows(), total=len(df_mt),
                           desc=f"Protocol B - {model_name}"):
            pid = str(row["pubmed_id"])
            if pid in done_ids:
                continue

            claim    = row["claim"]
            evidence = row["evidence"][:1000]

            p1 = MT_STAGE1_PROMPT.format(claim=claim, evidence=evidence)
            r1 = call_fn(p1, max_tokens=30)
            decision = parse_stage1(r1)
            time.sleep(sleep_sec)

            p2 = MT_STAGE2_PROMPT.format(decision=decision, claim=claim, evidence=evidence)
            r2 = call_fn(p2, max_tokens=30)
            error_prob = parse_stage2(r2)
            time.sleep(sleep_sec)

            p3 = MT_STAGE3_PROMPT.format(decision=decision, error_prob=f"{error_prob:.2f}",
                                          claim=claim, evidence=evidence)
            r3 = call_fn(p3, max_tokens=30)
            action = parse_stage3(r3)
            time.sleep(sleep_sec)

            s1_correct = int(decision == row["gold_label"])
            mf = compute_mf(s1_correct, error_prob)
            cr = compute_cr(s1_correct, action)

            writer.writerow({
                "pubmed_id":      pid,
                "gold_label":     row["gold_label"],
                "decision":       decision,
                "error_prob":     round(error_prob, 4),
                "action":         action,
                "stage1_correct": s1_correct,
                "mf":             round(mf, 4),
                "cr":             round(cr, 4),
                "raw_s1":         str(r1)[:100] if r1 else "",
                "raw_s2":         str(r2)[:100] if r2 else "",
                "raw_s3":         str(r3)[:100] if r3 else "",
            })
            f.flush()

    result_df = pd.read_csv(out_file)
    print(f"  {model_name} multi-turn done: {len(result_df)} samples")
    print(f"     Acc={result_df['stage1_correct'].mean():.4f}  "
          f"MF={result_df['mf'].mean():.4f}  CR={result_df['cr'].mean():.4f}")
    return out_file

print("Starting Protocol B (Multi-Turn) for all 3 models (200 samples each)...")
print("Estimated time: 20-30 minutes\n")

files_B = {}
for model_name, call_fn in MODEL_CALL_FN.items():
    files_B[model_name] = run_multi_turn(model_name, call_fn, MODEL_SLEEP[model_name])

print("\nProtocol B complete for all 3 models!")

In [ ]:
# CELL 10 — Compute all metrics with bootstrap confidence intervals

def full_metrics(df_m, model_name):
    n = len(df_m)
    acc   = df_m["stage1_correct"].mean()
    mf    = df_m["mf"].mean()
    cr    = df_m["cr"].mean()
    pdemc = compute_pdemc(acc, mf, cr)

    wrong = df_m[df_m["stage1_correct"] == 0]
    ccg   = (wrong["action"] == "COMMIT").mean() if len(wrong) > 0 else 0.0

    acc_lo,   acc_hi   = bootstrap_ci(df_m["stage1_correct"].tolist())
    mf_lo,    mf_hi    = bootstrap_ci(df_m["mf"].tolist())
    cr_lo,    cr_hi    = bootstrap_ci(df_m["cr"].tolist())
    pdemc_vals = [compute_pdemc(a, m, c)
                  for a, m, c in zip(df_m["stage1_correct"], df_m["mf"], df_m["cr"])]
    pdemc_lo, pdemc_hi = bootstrap_ci(pdemc_vals)

    per_class = {}
    for lbl in ["SUPPORTED", "REFUTED", "INCONCLUSIVE"]:
        sub = df_m[df_m["gold_label"] == lbl]
        per_class[lbl] = sub["stage1_correct"].mean() if len(sub) > 0 else 0.0

    action_dist = df_m["action"].value_counts(normalize=True).to_dict()
    overconf = (wrong["error_prob"] < 0.30).mean() if len(wrong) > 0 else 0.0

    return {
        "model": model_name, "n": n,
        "accuracy":  round(acc, 4),  "accuracy_ci":  (round(acc_lo,4),  round(acc_hi,4)),
        "mf":        round(mf, 4),   "mf_ci":        (round(mf_lo,4),   round(mf_hi,4)),
        "cr":        round(cr, 4),   "cr_ci":        (round(cr_lo,4),   round(cr_hi,4)),
        "pdemc":     round(pdemc,4), "pdemc_ci":     (round(pdemc_lo,4),round(pdemc_hi,4)),
        "ccg":       round(ccg, 4),
        "n_wrong":   len(wrong),
        "per_class": {k: round(v,4) for k,v in per_class.items()},
        "action_dist": {k: round(v,4) for k,v in action_dist.items()},
        "overconfidence": round(overconf, 4),
    }

dfs_A = {}
metrics_A = {}
for model_name in MODEL_CALL_FN:
    safe_name = model_name.replace("-","_").replace(".","_")
    dfs_A[model_name] = pd.read_csv(f"results_A_{safe_name}.csv")
    metrics_A[model_name] = full_metrics(dfs_A[model_name], model_name)

dfs_B = {}
metrics_B = {}
for model_name in MODEL_CALL_FN:
    safe_name = model_name.replace("-","_").replace(".","_")
    dfs_B[model_name] = pd.read_csv(f"results_B_{safe_name}.csv")
    metrics_B[model_name] = full_metrics(dfs_B[model_name], model_name)

print("="*70)
print("MAIN RESULTS - Protocol A (Single-Prompt), N=1000")
print("="*70)
print(f"{'Model':22s} {'Acc':>8} {'MF':>8} {'CR':>8} {'PDEMC':>8} {'CCG':>8}")
print("-"*70)
for m in metrics_A.values():
    print(f"{m['model']:22s} {m['accuracy']:>8.4f} {m['mf']:>8.4f} "
          f"{m['cr']:>8.4f} {m['pdemc']:>8.4f} {m['ccg']:>8.4f}")

print("\n" + "="*70)
print("PROTOCOL COMPARISON - Protocol A vs B (N=200)")
print("="*70)
print(f"{'Model':22s} {'Protocol':>14} {'Acc':>8} {'MF':>8} {'CR':>8} {'CCG':>8}")
print("-"*70)
for model_name in MODEL_CALL_FN:
    df_a200 = dfs_A[model_name].head(200)
    m_a = full_metrics(df_a200, model_name)
    m_b = metrics_B[model_name]
    print(f"{model_name:22s} {'A (single)':>14} {m_a['accuracy']:>8.4f} {m_a['mf']:>8.4f} "
          f"{m_a['cr']:>8.4f} {m_a['ccg']:>8.4f}")
    print(f"{'':22s} {'B (multi-turn)':>14} {m_b['accuracy']:>8.4f} {m_b['mf']:>8.4f} "
          f"{m_b['cr']:>8.4f} {m_b['ccg']:>8.4f}")
    print("-"*70)

print("\nAll metrics computed successfully")

In [ ]:
# CELL 11 — Threshold sweep 0.10-0.50

THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50]
threshold_results = {}

print("THRESHOLD SWEEP")
print("="*75)

for model_name, df_m in dfs_A.items():
    wrong = df_m[df_m["stage1_correct"] == 0]
    threshold_results[model_name] = []
    print(f"\n{model_name}:")
    print(f"  {'Threshold':>10} {'Baseline_CR':>12} {'Baseline_CCG':>13} {'Abstain_Wrong%':>15}")

    for t in THRESHOLDS:
        correct = 0
        for _, r in df_m.iterrows():
            bl_action = "COMMIT" if r["error_prob"] < t else "ABSTAIN"
            if r["stage1_correct"]:
                correct += 1 if bl_action == "COMMIT" else 0
            else:
                correct += 0 if bl_action == "COMMIT" else 1
        bl_cr  = correct / len(df_m)
        bl_ccg = (wrong["error_prob"] < t).mean() if len(wrong) > 0 else 0.0
        abstain_wrong = (wrong["error_prob"] >= t).mean() if len(wrong) > 0 else 0.0

        marker = " <- original" if t == 0.30 else ""
        print(f"  {t:>10.2f} {bl_cr:>12.4f} {bl_ccg:>13.4f} {abstain_wrong*100:>14.1f}%{marker}")
        threshold_results[model_name].append({
            "threshold": t, "baseline_cr": round(bl_cr,4),
            "baseline_ccg": round(bl_ccg,4),
            "abstain_wrong_pct": round(abstain_wrong*100,2)
        })

print("\nThreshold sweep complete")

In [ ]:
# CELL 12 — PDEMC weight sensitivity analysis

WEIGHT_CONFIGS = [
    (0.30, 0.30, 0.40, "30/30/40 (original)"),
    (0.33, 0.33, 0.34, "33/33/34 (equal)"),
    (0.25, 0.25, 0.50, "25/25/50 (control-heavy)"),
    (0.50, 0.25, 0.25, "50/25/25 (accuracy-heavy)"),
    (0.20, 0.30, 0.50, "20/30/50 (monitor+control)"),
    (0.40, 0.20, 0.40, "40/20/40 (accuracy+control)"),
]

print("PDEMC WEIGHT SENSITIVITY ANALYSIS")
print("="*80)
header = f"{'Weights (Acc/MF/CR)':>30}"
for m in metrics_A: header += f" {m:>18}"
print(header)
print("-"*80)

sensitivity_results = []
for w_acc, w_mf, w_cr, label in WEIGHT_CONFIGS:
    row_str = f"  {label:>28}"
    row_data = {"weights": label}
    rankings = []
    for model_name, m in metrics_A.items():
        p = compute_pdemc(m["accuracy"], m["mf"], m["cr"], w_acc, w_mf, w_cr)
        row_str += f" {p:>18.4f}"
        row_data[model_name] = round(p, 4)
        rankings.append((p, model_name))
    rankings.sort(reverse=True)
    row_data["ranking"] = " > ".join([r[1] for r in rankings])
    print(row_str)
    sensitivity_results.append(row_data)

all_rankings = [r["ranking"] for r in sensitivity_results]
unique_rankings = set(all_rankings)
print(f"\nRanking stability: {len(unique_rankings)} unique ranking(s) across {len(WEIGHT_CONFIGS)} configurations")
if len(unique_rankings) == 1:
    print("Rankings are STABLE across all weight configurations")
else:
    print("Rankings change under some configurations - see details above")

print("\nSensitivity analysis complete")

In [ ]:
# CELL 13 — Additional baselines comparison

def compute_baselines(df_m, model_name):
    n = len(df_m)
    wrong = df_m[df_m["stage1_correct"] == 0]
    acc   = df_m["stage1_correct"].mean()
    mf    = df_m["mf"].mean()

    ac_cr    = acc
    ac_ccg   = 1.0
    ac_pdemc = compute_pdemc(acc, mf, ac_cr)

    aa_cr    = 1.0 - acc
    aa_ccg   = 0.0
    aa_pdemc = compute_pdemc(acc, mf, aa_cr)

    np.random.seed(RANDOM_SEED)
    rand_acts = np.random.choice(["COMMIT", "ABSTAIN"], size=n)
    rand_cr = sum(
        1 for i, (_, r) in enumerate(df_m.iterrows())
        if (r["stage1_correct"] and rand_acts[i] == "COMMIT") or
           (not r["stage1_correct"] and rand_acts[i] != "COMMIT")
    ) / n
    rand_ccg = sum(
        1 for i, (_, r) in enumerate(df_m.iterrows())
        if not r["stage1_correct"] and rand_acts[i] == "COMMIT"
    ) / max(1, len(wrong))
    rand_pdemc = compute_pdemc(acc, mf, rand_cr)

    t_correct = sum(
        1 for _, r in df_m.iterrows()
        if (r["stage1_correct"] and r["error_prob"] < 0.30) or
           (not r["stage1_correct"] and r["error_prob"] >= 0.30)
    )
    t_cr    = t_correct / n
    t_ccg   = (wrong["error_prob"] < 0.30).mean() if len(wrong) > 0 else 0.0
    t_pdemc = compute_pdemc(acc, mf, t_cr)

    m = metrics_A[model_name]

    return {
        "Always COMMIT":      {"cr": round(ac_cr,4),   "ccg": round(ac_ccg,4),   "pdemc": round(ac_pdemc,4)},
        "Always ABSTAIN":     {"cr": round(aa_cr,4),   "ccg": round(aa_ccg,4),   "pdemc": round(aa_pdemc,4)},
        "Random Action":      {"cr": round(rand_cr,4), "ccg": round(rand_ccg,4), "pdemc": round(rand_pdemc,4)},
        "Threshold (p<0.30)": {"cr": round(t_cr,4),    "ccg": round(t_ccg,4),    "pdemc": round(t_pdemc,4)},
        f"{model_name} (SOEA)": {"cr": round(m["cr"],4), "ccg": round(m["ccg"],4), "pdemc": round(m["pdemc"],4)},
    }

all_baselines = {}
for model_name in MODEL_CALL_FN:
    all_baselines[model_name] = compute_baselines(dfs_A[model_name], model_name)

print("BASELINES COMPARISON")
print("="*75)
for model_name, baselines in all_baselines.items():
    print(f"\n{model_name}:")
    print(f"  {'Method':>30} {'CR':>8} {'CCG':>8} {'PDEMC':>8}")
    print("  " + "-"*55)
    for bl_name, vals in baselines.items():
        marker = " <-" if "SOEA" in bl_name else ""
        print(f"  {bl_name:>30} {vals['cr']:>8.4f} {vals['ccg']:>8.4f} {vals['pdemc']:>8.4f}{marker}")

print("\nBaselines comparison complete")

In [ ]:
# CELL 14 — Save ALL results to JSON

all_results = {
    "protocol_A": {k: v for k, v in metrics_A.items()},
    "protocol_B": {k: v for k, v in metrics_B.items()},
    "threshold_sweep": threshold_results,
    "sensitivity_analysis": sensitivity_results,
    "baselines": all_baselines,
    "dataset_metadata": metadata,
}

with open("soea_v2_all_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("All results saved to soea_v2_all_results.json")
print(f"  Protocol A: {len(metrics_A)} models x {N_SAMPLES} samples")
print(f"  Protocol B: {len(metrics_B)} models x 200 samples")
print(f"  Threshold sweep: {len(THRESHOLDS)} thresholds x {len(MODEL_CALL_FN)} models")
print(f"  Sensitivity: {len(WEIGHT_CONFIGS)} weight configs")
print(f"  Baselines: 4 baselines x {len(MODEL_CALL_FN)} models")

In [ ]:
# CELL 15 — Generate all 5 figures for the paper

model_names = list(MODEL_CALL_FN.keys())

# FIGURE 1: PDEMC Component Breakdown + Per-Class Accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metric_labels = ["Accuracy", "Monitoring\nFidelity", "Control\nRationality", "PDEMC"]
x = np.arange(len(metric_labels))
w = 0.25
ax = axes[0]
for i, mn in enumerate(model_names):
    m = metrics_A[mn]
    vals = [m["accuracy"], m["mf"], m["cr"], m["pdemc"]]
    bars = ax.bar(x + (i - 1) * w, vals, w, label=mn, color=MODEL_COLORS[mn], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
ax.set_title("(a) PDEMC Component Breakdown"); ax.legend(fontsize=9)
ax.axhline(y=0.5, color="gray", linestyle=":", alpha=0.5)

ax2 = axes[1]
classes = ["SUPPORTED", "REFUTED", "INCONCLUSIVE"]
for i, mn in enumerate(model_names):
    m = metrics_A[mn]
    vals = [m["per_class"].get(c, 0) for c in classes]
    ax2.bar(np.arange(len(classes)) + (i-1)*w, vals, w, label=mn, color=MODEL_COLORS[mn], alpha=0.85)
ax2.set_xticks(np.arange(len(classes))); ax2.set_xticklabels(classes)
ax2.set_ylim(0, 1.05); ax2.set_ylabel("Accuracy")
ax2.set_title("(b) Per-Class Accuracy"); ax2.legend(fontsize=9)
plt.tight_layout()
plt.savefig("fig1_pdemc_breakdown.pdf", bbox_inches="tight", dpi=300)
plt.savefig("fig1_pdemc_breakdown.png", bbox_inches="tight", dpi=300)
plt.show(); print("Figure 1 saved")

# FIGURE 2: Control Collapse Gap
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, mn in enumerate(model_names):
    df_m = dfs_A[mn]
    wrong = df_m[df_m["stage1_correct"] == 0]
    commit_w = (wrong["action"] == "COMMIT").sum()
    safe_w   = len(wrong) - commit_w
    ccg_pct  = commit_w / len(wrong) * 100 if len(wrong) > 0 else 0
    axes[idx].pie([commit_w, safe_w],
        labels=[f"COMMIT (Collapse)\n{ccg_pct:.1f}%", f"Safe Action\n{100-ccg_pct:.1f}%"],
        colors=["#F44336", "#4CAF50"], startangle=90, textprops={"fontsize": 10})
    axes[idx].set_title(f"{mn}\nCCG = {ccg_pct:.1f}%", fontsize=11)
plt.suptitle("Figure 2: Control Collapse Gap - Actions When Model is Wrong", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("fig2_control_collapse.pdf", bbox_inches="tight", dpi=300)
plt.savefig("fig2_control_collapse.png", bbox_inches="tight", dpi=300)
plt.show(); print("Figure 2 saved")

# FIGURE 3: Threshold Sweep
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for mn in model_names:
    ts   = [r["threshold"] for r in threshold_results[mn]]
    crs  = [r["baseline_cr"] for r in threshold_results[mn]]
    ccgs = [r["baseline_ccg"] for r in threshold_results[mn]]
    ax1.plot(ts, crs,  marker="o", label=mn, color=MODEL_COLORS[mn])
    ax2.plot(ts, ccgs, marker="s", label=mn, color=MODEL_COLORS[mn], linestyle="--")
for ax in [ax1, ax2]:
    ax.axvline(x=0.30, color="red", linestyle=":", alpha=0.7, label="Threshold=0.30")
    ax.set_xticks(THRESHOLDS); ax.legend(fontsize=9)
ax1.set_xlabel("Error Probability Threshold"); ax1.set_ylabel("Baseline CR")
ax1.set_title("(a) Baseline CR vs. Threshold")
ax2.set_xlabel("Error Probability Threshold"); ax2.set_ylabel("Baseline CCG")
ax2.set_title("(b) Baseline CCG vs. Threshold")
plt.suptitle("Figure 3: Threshold Sweep (0.10-0.50)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("fig3_threshold_sweep.pdf", bbox_inches="tight", dpi=300)
plt.savefig("fig3_threshold_sweep.png", bbox_inches="tight", dpi=300)
plt.show(); print("Figure 3 saved")

# FIGURE 4: Sensitivity Analysis
fig, ax = plt.subplots(figsize=(12, 4))
wlabels = [r["weights"] for r in sensitivity_results]
x = np.arange(len(wlabels)); w = 0.25
for i, mn in enumerate(model_names):
    pvals = [r[mn] for r in sensitivity_results]
    ax.bar(x + (i-1)*w, pvals, w, label=mn, color=MODEL_COLORS[mn], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(wlabels, fontsize=8, rotation=15)
ax.set_ylabel("PDEMC Score")
ax.set_title("Figure 4: PDEMC Sensitivity to Weight Configurations")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("fig4_sensitivity.pdf", bbox_inches="tight", dpi=300)
plt.savefig("fig4_sensitivity.png", bbox_inches="tight", dpi=300)
plt.show(); print("Figure 4 saved")

# FIGURE 5: Protocol A vs B comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metric_keys   = ["accuracy", "cr", "ccg"]
metric_titles = ["Accuracy", "Control Rationality (CR)", "Control Collapse Gap (CCG)"]
for ax_i, (mk, mt) in enumerate(zip(metric_keys, metric_titles)):
    ax = axes[ax_i]; x = np.arange(len(model_names)); w = 0.35
    vals_a = [full_metrics(dfs_A[mn].head(200), mn)[mk] for mn in model_names]
    vals_b = [metrics_B[mn][mk] for mn in model_names]
    ax.bar(x - w/2, vals_a, w, label="Protocol A (single)",    color="#607D8B", alpha=0.85)
    ax.bar(x + w/2, vals_b, w, label="Protocol B (multi-turn)", color="#FF5722", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels([m.replace("-","\n") for m in model_names], fontsize=9)
    ax.set_title(mt); ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
plt.suptitle("Figure 5: Single-Prompt vs. Multi-Turn Protocol Comparison (N=200)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("fig5_protocol_comparison.pdf", bbox_inches="tight", dpi=300)
plt.savefig("fig5_protocol_comparison.png", bbox_inches="tight", dpi=300)
plt.show(); print("Figure 5 saved")

print("\nALL 5 FIGURES GENERATED")

In [ ]:
# CELL 16 — Print paper-ready tables

print("="*70)
print("TABLE 1: Main Results - Protocol A (N=1000, seed=42)")
print("="*70)
print(f"{'Model':22s} {'Acc (95% CI)':>18} {'MF':>8} {'CR (95% CI)':>16} {'PDEMC':>8} {'CCG':>8}")
print("-"*80)
for mn in model_names:
    m = metrics_A[mn]
    acc_ci = m["accuracy_ci"]
    cr_ci  = m["cr_ci"]
    print(f"{mn:22s} "
          f"{m['accuracy']:.3f} [{acc_ci[0]:.3f}-{acc_ci[1]:.3f}] "
          f"{m['mf']:.3f} "
          f"{m['cr']:.3f} [{cr_ci[0]:.3f}-{cr_ci[1]:.3f}] "
          f"{m['pdemc']:.3f} "
          f"{m['ccg']*100:.1f}%")

print("\n" + "="*70)
print("TABLE 2: Per-Class Accuracy")
print("="*70)
print(f"{'Model':22s} {'SUPPORTED':>12} {'REFUTED':>10} {'INCONCLUSIVE':>14}")
print("-"*60)
for mn in model_names:
    m = metrics_A[mn]
    print(f"{mn:22s} "
          f"{m['per_class'].get('SUPPORTED',0):>12.3f} "
          f"{m['per_class'].get('REFUTED',0):>10.3f} "
          f"{m['per_class'].get('INCONCLUSIVE',0):>14.3f}")

print("\n" + "="*70)
print("TABLE 3: Baselines Comparison")
print("="*70)
for model_name in model_names:
    print(f"\n{model_name}:")
    print(f"  {'Method':>30} {'CR':>8} {'CCG':>8} {'PDEMC':>8}")
    print("  " + "-"*55)
    for bl_name, vals in all_baselines[model_name].items():
        marker = " <-" if "SOEA" in bl_name else ""
        print(f"  {bl_name:>30} {vals['cr']:>8.3f} {vals['ccg']:>8.3f} {vals['pdemc']:>8.3f}{marker}")

print("\n" + "="*70)
print("TABLE 4: Threshold Sweep (GPT-4.1-mini)")
print("="*70)
print(f"{'Threshold':>10} {'Baseline CR':>12} {'Baseline CCG':>13} {'Abstain-on-Wrong%':>18}")
print("-"*60)
for r in threshold_results["GPT-4.1-mini"]:
    marker = " <-" if r["threshold"] == 0.30 else ""
    print(f"{r['threshold']:>10.2f} {r['baseline_cr']:>12.3f} "
          f"{r['baseline_ccg']:>13.3f} {r['abstain_wrong_pct']:>17.1f}%{marker}")

print("\n" + "="*70)
print("TABLE 5: Protocol Comparison (N=200)")
print("="*70)
print(f"{'Model':22s} {'Protocol':>15} {'Acc':>8} {'MF':>8} {'CR':>8} {'CCG':>8}")
print("-"*70)
for mn in model_names:
    df_a200 = dfs_A[mn].head(200)
    m_a = full_metrics(df_a200, mn)
    m_b = metrics_B[mn]
    print(f"{mn:22s} {'Single-prompt':>15} {m_a['accuracy']:>8.3f} {m_a['mf']:>8.3f} "
          f"{m_a['cr']:>8.3f} {m_a['ccg']:>8.3f}")
    print(f"{'':22s} {'Multi-turn':>15} {m_b['accuracy']:>8.3f} {m_b['mf']:>8.3f} "
          f"{m_b['cr']:>8.3f} {m_b['ccg']:>8.3f}")
    print("-"*70)

print("\nAll paper-ready tables printed")

In [ ]:
# CELL 17 — Download all results as ZIP
from google.colab import files

output_files = [
    "soea_v2_all_results.json",
    "dataset_metadata_v2.json",
    "results_A_GPT_4_1_mini.csv",
    "results_A_Llama_3_3_70b.csv",
    "results_A_Gemini_2_5_Flash.csv",
    "results_B_GPT_4_1_mini.csv",
    "results_B_Llama_3_3_70b.csv",
    "results_B_Gemini_2_5_Flash.csv",
    "fig1_pdemc_breakdown.pdf",   "fig1_pdemc_breakdown.png",
    "fig2_control_collapse.pdf",  "fig2_control_collapse.png",
    "fig3_threshold_sweep.pdf",   "fig3_threshold_sweep.png",
    "fig4_sensitivity.pdf",       "fig4_sensitivity.png",
    "fig5_protocol_comparison.pdf", "fig5_protocol_comparison.png",
]

with zipfile.ZipFile("SOEA_Plus_V2_Results.zip", "w") as zf:
    for fname in output_files:
        if os.path.exists(fname):
            zf.write(fname)
            print(f"  Added: {fname}")
        else:
            print(f"  Missing: {fname}")

print("\nDownloading ZIP file...")
files.download("SOEA_Plus_V2_Results.zip")
print("\nDONE! Check your Downloads folder for SOEA_Plus_V2_Results.zip")
print("Send this file back to update the paper with the new results.")

In [ ]:
# ============================================================
# SOEA-Plus V2 Result Auditor
# Purpose:
# 1) Check Gemini collapse / parsing problems
# 2) Generate confusion matrices
# 3) Compute invalid-output rates
# 4) Compare labels, actions, probabilities
# 5) Save clean audit tables and figures
# ============================================================

import os
import re
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# -----------------------------
# Config
# -----------------------------
RESULT_DIRS = [".", "./figures", "./results", "./outputs"]
OUT_DIR = "./soea_audit_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

VALID_LABELS = ["SUPPORTED", "REFUTED", "INCONCLUSIVE"]
VALID_ACTIONS = ["COMMIT", "REVISE", "ABSTAIN", "SEEK_EVIDENCE"]

# -----------------------------
# Helpers
# -----------------------------
def find_result_files():
    files = []
    for d in RESULT_DIRS:
        if os.path.exists(d):
            files += glob.glob(os.path.join(d, "*.csv"))
            files += glob.glob(os.path.join(d, "*.json"))
    return sorted(set(files))

def normalize_label(x):
    if pd.isna(x):
        return None
    s = str(x).upper().strip()
    s = re.sub(r"[^A-Z_ ]", "", s)
    if "SUPPORTED" in s or s in ["YES", "TRUE"]:
        return "SUPPORTED"
    if "REFUTED" in s or s in ["NO", "FALSE"]:
        return "REFUTED"
    if "INCONCLUSIVE" in s or "UNCERTAIN" in s or "UNKNOWN" in s or "MAYBE" in s:
        return "INCONCLUSIVE"
    return None

def normalize_action(x):
    if pd.isna(x):
        return None
    s = str(x).upper().strip()
    s = re.sub(r"[^A-Z_ ]", "", s)
    if "SEEK" in s:
        return "SEEK_EVIDENCE"
    if "ABSTAIN" in s:
        return "ABSTAIN"
    if "REVISE" in s:
        return "REVISE"
    if "COMMIT" in s:
        return "COMMIT"
    return None

def to_float_prob(x):
    try:
        v = float(x)
        if 0 <= v <= 1:
            return v
        return np.nan
    except Exception:
        return np.nan

def detect_columns(df):
    cols = {c.lower(): c for c in df.columns}
    
    label_col = None
    pred_col = None
    action_col = None
    prob_col = None
    raw_col = None
    
    for c in df.columns:
        lc = c.lower()
        if label_col is None and any(k in lc for k in ["gold", "true", "label", "answer"]):
            if "pred" not in lc and "model" not in lc:
                label_col = c
        if pred_col is None and any(k in lc for k in ["pred", "decision", "model_label", "prediction"]):
            pred_col = c
        if action_col is None and "action" in lc:
            action_col = c
        if prob_col is None and any(k in lc for k in ["prob", "error_probability", "p_error", "confidence"]):
            prob_col = c
        if raw_col is None and any(k in lc for k in ["raw", "response", "output", "text"]):
            raw_col = c
    
    return label_col, pred_col, action_col, prob_col, raw_col

def load_csv_files():
    csvs = [f for f in find_result_files() if f.endswith(".csv")]
    loaded = {}
    for f in csvs:
        try:
            df = pd.read_csv(f)
            loaded[f] = df
        except Exception as e:
            print(f"Could not read {f}: {e}")
    return loaded

def infer_model_name(path):
    p = path.lower()
    if "gemini" in p:
        return "Gemini-2.5-Flash"
    if "llama" in p:
        return "Llama-3.3-70B"
    if "gpt" in p or "openai" in p:
        return "GPT-4.1-mini"
    return os.path.basename(path)

def audit_dataframe(df, name):
    label_col, pred_col, action_col, prob_col, raw_col = detect_columns(df)
    
    print("\n" + "="*90)
    print(f"FILE/MODEL: {name}")
    print("="*90)
    print("Detected columns:")
    print({
        "gold_label": label_col,
        "prediction": pred_col,
        "action": action_col,
        "probability": prob_col,
        "raw_output": raw_col,
    })
    
    work = df.copy()
    
    if label_col:
        work["_gold"] = work[label_col].apply(normalize_label)
    else:
        work["_gold"] = None
        
    if pred_col:
        work["_pred"] = work[pred_col].apply(normalize_label)
    else:
        work["_pred"] = None
        
    if action_col:
        work["_action"] = work[action_col].apply(normalize_action)
    else:
        work["_action"] = None
        
    if prob_col:
        work["_p_error"] = work[prob_col].apply(to_float_prob)
    else:
        work["_p_error"] = np.nan
    
    n = len(work)
    invalid_pred = work["_pred"].isna().mean() if n else np.nan
    invalid_action = work["_action"].isna().mean() if n else np.nan
    invalid_prob = work["_p_error"].isna().mean() if n else np.nan
    
    print(f"\nN = {n}")
    print(f"Invalid prediction rate: {invalid_pred:.2%}")
    print(f"Invalid action rate:     {invalid_action:.2%}")
    print(f"Invalid probability rate:{invalid_prob:.2%}")
    
    print("\nPrediction distribution:")
    print(work["_pred"].value_counts(dropna=False))
    
    print("\nGold label distribution:")
    print(work["_gold"].value_counts(dropna=False))
    
    print("\nAction distribution:")
    print(work["_action"].value_counts(dropna=False))
    
    if work["_gold"].notna().any() and work["_pred"].notna().any():
        valid = work[work["_gold"].notna() & work["_pred"].notna()]
        acc = (valid["_gold"] == valid["_pred"]).mean()
        print(f"\nParsed Accuracy = {acc:.4f}")
        
        print("\nClassification report:")
        print(classification_report(
            valid["_gold"],
            valid["_pred"],
            labels=VALID_LABELS,
            zero_division=0
        ))
        
        cm = confusion_matrix(valid["_gold"], valid["_pred"], labels=VALID_LABELS)
        cm_df = pd.DataFrame(cm, index=VALID_LABELS, columns=VALID_LABELS)
        print("\nConfusion matrix:")
        print(cm_df)
        
        # Save confusion matrix
        safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", name)
        cm_df.to_csv(os.path.join(OUT_DIR, f"confusion_matrix_{safe_name}.csv"))
        
        # Plot confusion matrix
        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title(f"Confusion Matrix: {name}")
        plt.xticks(range(len(VALID_LABELS)), VALID_LABELS, rotation=45, ha="right")
        plt.yticks(range(len(VALID_LABELS)), VALID_LABELS)
        plt.xlabel("Predicted")
        plt.ylabel("Gold")
        for i in range(len(VALID_LABELS)):
            for j in range(len(VALID_LABELS)):
                plt.text(j, i, cm[i, j], ha="center", va="center")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"confusion_matrix_{safe_name}.png"), dpi=300)
        plt.show()
    
    # Show suspicious examples
    print("\nSample raw/problematic rows:")
    suspicious = work[
        work["_pred"].isna() |
        work["_action"].isna() |
        work["_p_error"].isna()
    ].head(10)
    
    if len(suspicious) == 0:
        suspicious = work.head(10)
        
    display_cols = []
    for c in [label_col, pred_col, action_col, prob_col, raw_col]:
        if c and c not in display_cols:
            display_cols.append(c)
    display_cols += ["_gold", "_pred", "_action", "_p_error"]
    display_cols = [c for c in display_cols if c in work.columns]
    
    display(suspicious[display_cols])
    
    # Save audited file
    safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", name)
    work.to_csv(os.path.join(OUT_DIR, f"audited_{safe_name}.csv"), index=False)
    
    return {
        "name": name,
        "n": n,
        "invalid_pred": invalid_pred,
        "invalid_action": invalid_action,
        "invalid_prob": invalid_prob,
        "pred_distribution": work["_pred"].value_counts(dropna=False).to_dict(),
        "action_distribution": work["_action"].value_counts(dropna=False).to_dict(),
    }

# -----------------------------
# Run audit
# -----------------------------
loaded = load_csv_files()

print(f"Found {len(loaded)} CSV files:")
for f in loaded:
    print(" -", f)

summaries = []

for path, df in loaded.items():
    model_name = infer_model_name(path)
    
    # only audit likely result files
    lower = os.path.basename(path).lower()
    if any(k in lower for k in ["result", "gpt", "llama", "gemini", "protocol"]):
        summary = audit_dataframe(df, model_name + "__" + os.path.basename(path))
        summaries.append(summary)

summary_df = pd.DataFrame(summaries)
summary_path = os.path.join(OUT_DIR, "audit_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\n" + "="*90)
print("AUDIT SUMMARY SAVED TO:", summary_path)
print("="*90)
display(summary_df)

In [ ]:
# ============================================================
# SOEA-Plus V2: Manual Error Analysis Export
# Purpose:
# - Create a 100-case qualitative audit sample
# - Categorize failure types automatically
# - Export a manual validation sheet
# - Produce paper-ready summary tables
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

# ----------------------------
# Config
# ----------------------------
RANDOM_SEED = 42
AUDIT_N = 100
OUT_DIR = "soea_manual_audit"
os.makedirs(OUT_DIR, exist_ok=True)

VALID_LABELS = ["SUPPORTED", "REFUTED", "INCONCLUSIVE"]
SAFE_ACTIONS = ["ABSTAIN", "SEEK_EVIDENCE", "REVISE"]

# ----------------------------
# Load result files
# ----------------------------
csv_files = glob.glob("results_A_*.csv") + glob.glob("results_B_*.csv")

if not csv_files:
    raise FileNotFoundError("No results_A_*.csv or results_B_*.csv files found in the current folder.")

all_dfs = []

for file in csv_files:
    df = pd.read_csv(file)
    
    model = file.replace("results_A_", "").replace("results_B_", "")
    model = model.replace(".csv", "").replace("_", "-")
    
    protocol = "A_single_prompt" if "results_A_" in file else "B_multi_turn"
    
    df["model"] = model
    df["protocol"] = protocol
    df["source_file"] = file
    
    all_dfs.append(df)

data = pd.concat(all_dfs, ignore_index=True)

# ----------------------------
# Normalize required columns
# ----------------------------
required_cols = ["gold_label", "decision", "action", "error_prob"]

missing = [c for c in required_cols if c not in data.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

data["gold_label"] = data["gold_label"].astype(str).str.upper().str.strip()
data["decision"] = data["decision"].astype(str).str.upper().str.strip()
data["action"] = data["action"].astype(str).str.upper().str.strip()
data["error_prob"] = pd.to_numeric(data["error_prob"], errors="coerce")

data["is_correct"] = data["gold_label"] == data["decision"]
data["is_wrong"] = ~data["is_correct"]
data["is_commit"] = data["action"] == "COMMIT"
data["is_safe_action"] = data["action"].isin(SAFE_ACTIONS)

# ----------------------------
# Failure type classifier
# ----------------------------
def classify_failure(row):
    correct = row["is_correct"]
    action = row["action"]
    p = row["error_prob"]
    gold = row["gold_label"]
    pred = row["decision"]
    
    if pd.isna(p):
        return "Invalid probability"
    
    if correct and action == "COMMIT":
        return "Safe correct commit"
    
    if correct and action in SAFE_ACTIONS:
        return "Over-cautious safe action"
    
    if not correct and action == "COMMIT" and p < 0.30:
        return "High-confidence wrong commit"
    
    if not correct and action == "COMMIT" and p >= 0.30:
        return "Uncertainty ignored"
    
    if not correct and action in SAFE_ACTIONS:
        return "Safe response to wrong decision"
    
    if pred == "INCONCLUSIVE" and action == "COMMIT":
        return "Inconclusive commit paradox"
    
    if pred == "INCONCLUSIVE":
        return "Inconclusive bias"
    
    return "Other"

data["failure_type"] = data.apply(classify_failure, axis=1)

# ----------------------------
# Priority sampling
# ----------------------------
# We want a useful audit sheet, not a purely random one.
# Include dangerous and scientifically informative cases first.
priority_types = [
    "High-confidence wrong commit",
    "Uncertainty ignored",
    "Inconclusive commit paradox",
    "Over-cautious safe action",
    "Safe response to wrong decision",
    "Safe correct commit"
]

audit_parts = []

per_type_n = max(5, AUDIT_N // len(priority_types))

for ft in priority_types:
    subset = data[data["failure_type"] == ft]
    if len(subset) > 0:
        audit_parts.append(
            subset.sample(
                n=min(per_type_n, len(subset)),
                random_state=RANDOM_SEED
            )
        )

audit_df = pd.concat(audit_parts, ignore_index=True) if audit_parts else pd.DataFrame()

# Fill remaining with random samples
remaining_n = AUDIT_N - len(audit_df)

if remaining_n > 0:
    remaining_pool = data.drop(index=audit_df.index, errors="ignore")
    extra = remaining_pool.sample(
        n=min(remaining_n, len(remaining_pool)),
        random_state=RANDOM_SEED
    )
    audit_df = pd.concat([audit_df, extra], ignore_index=True)

audit_df = audit_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# ----------------------------
# Add manual annotation columns
# ----------------------------
audit_df["manual_control_judgment"] = ""
audit_df["better_action_if_any"] = ""
audit_df["manual_notes"] = ""

# Keep useful columns
preferred_cols = [
    "model",
    "protocol",
    "gold_label",
    "decision",
    "error_prob",
    "action",
    "is_correct",
    "failure_type",
    "manual_control_judgment",
    "better_action_if_any",
    "manual_notes",
    "source_file"
]

# Add claim/evidence/raw if available
for optional in ["claim", "evidence", "question", "context", "raw_response", "raw_s1", "raw_s2", "raw_s3"]:
    if optional in audit_df.columns and optional not in preferred_cols:
        preferred_cols.insert(2, optional)

audit_df = audit_df[[c for c in preferred_cols if c in audit_df.columns]]

# ----------------------------
# Save audit files
# ----------------------------
csv_path = os.path.join(OUT_DIR, "manual_audit_100_cases.csv")
xlsx_path = os.path.join(OUT_DIR, "manual_audit_100_cases.xlsx")

audit_df.to_csv(csv_path, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    audit_df.to_excel(writer, index=False, sheet_name="Manual_Audit")
    
    worksheet = writer.sheets["Manual_Audit"]
    worksheet.freeze_panes = "A2"
    
    for col in worksheet.columns:
        max_len = 0
        col_letter = col[0].column_letter
        for cell in col:
            try:
                max_len = max(max_len, len(str(cell.value)))
            except:
                pass
        worksheet.column_dimensions[col_letter].width = min(max_len + 2, 45)

# ----------------------------
# Failure summary
# ----------------------------
failure_summary = (
    data.groupby(["model", "protocol", "failure_type"])
    .size()
    .reset_index(name="count")
)

failure_summary["percent_within_model_protocol"] = (
    failure_summary.groupby(["model", "protocol"])["count"]
    .transform(lambda x: 100 * x / x.sum())
    .round(2)
)

failure_summary_path = os.path.join(OUT_DIR, "failure_type_summary.csv")
failure_summary.to_csv(failure_summary_path, index=False)

# ----------------------------
# Paper-ready summary
# ----------------------------
summary_lines = []
summary_lines.append("SOEA-Plus Manual Error Analysis Summary")
summary_lines.append("=" * 60)
summary_lines.append(f"Total evaluated rows loaded: {len(data)}")
summary_lines.append(f"Manual audit sample size: {len(audit_df)}")
summary_lines.append("")
summary_lines.append("Failure Type Distribution:")
summary_lines.append(str(data["failure_type"].value_counts()))
summary_lines.append("")
summary_lines.append("Model / Protocol Summary:")
summary_lines.append(
    str(
        data.groupby(["model", "protocol"])
        .agg(
            N=("gold_label", "count"),
            Accuracy=("is_correct", "mean"),
            Commit_Rate=("is_commit", "mean"),
            Safe_Action_Rate=("is_safe_action", "mean"),
            Mean_Error_Prob=("error_prob", "mean")
        )
        .round(4)
    )
)

summary_text = "\n".join(summary_lines)

summary_path = os.path.join(OUT_DIR, "paper_ready_audit_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)

print("\nSaved files:")
print(f"- {csv_path}")
print(f"- {xlsx_path}")
print(f"- {failure_summary_path}")
print(f"- {summary_path}")

display(audit_df.head(10))
display(failure_summary.head(20))